# Create the knowledge question bank

**Goal:** Prepare source packets, inspect generation instructions and validate new candidates.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Read the source plan

The study used 1,000 ICSD-3-TR and 215 AASM questions. Candidate creation precedes inspection of those saved results. Supply your own licensed PDFs under outputs/private/sources/knowledge. Physical PDF page numbers in source packets start at one.

In [ ]:
from sleepinn_study.databank import source_packet, generation_prompt, parse_candidates
RUN_GENERATION = False
source_folder = ROOT / "outputs/private/sources/knowledge"
print("Local source folder:", source_folder)


## 3. Inspect the generation prompts

The saved ICSD instructions were a shared instruction block covering all three question types, followed by the current source unit. They are included verbatim below. The type-specific helper is a reusable prompt for new work; it is not presented as three historical prompts.

In [ ]:
instructions = ROOT / "data/config/databank/prompts/icsd3_historical_generation_instructions.txt"
if instructions.exists():
    print(instructions.read_text(encoding="utf-8"))
else:
    for p in (ROOT / "data/config/databank/prompts").glob("*instructions*"):
        print(p.read_text(encoding="utf-8"))
example_packet = {"source_pdf": "example.pdf", "pages": [{"page": 1, "text": "Illustrative source text goes here."}]}
for question_type in ["mcq", "free_text_qa", "criteria_qa"]:
    print("\nTYPE:", question_type)
    print(generation_prompt(example_packet, question_type, 2))

## 4. Generate and validate a new batch

Only enable this cell after choosing a real PDF and page range. The API key is read from the environment. Candidate validation checks schema, source pages and literal evidence quotations; human review follows in notebook 02.

In [ ]:
if RUN_GENERATION:
    from sleepinn_study.providers import request_json, answer_text
    from sleepinn_study.io import write_json
    pdf = source_folder / "your_source.pdf"
    packet = source_packet(pdf, first_page=1, last_page=3)
    prompt = generation_prompt(packet, "mcq", 2)
    payload = {"contents": [{"role": "user", "parts": [{"text": prompt}]}],
               "generationConfig": {"maxOutputTokens": 8192, "responseMimeType": "application/json"}}
    raw = request_json("gemini", "gemini-3.8-flash", payload,
                       ROOT / "outputs/authoring/new_batch", enabled=True)
    candidates = parse_candidates(answer_text("gemini", raw), packet, "mcq")
    write_json(ROOT / "outputs/authoring/new_candidates.json", candidates)
else:
    print("Generation disabled. No network request was made.")

## 5. Inspect the saved candidate set

These counts describe the output of candidate generation, before AI revision. They are not inputs to the preceding prompt example.

In [ ]:
from sleepinn_study.databank import validate_items
candidates = read_jsonl(ROOT / "data/candidates/knowledge.jsonl")
display(validate_items(candidates))
display(pd.DataFrame(candidates).groupby(["source_pdf", "item_type"]).size().rename("questions"))